## Case study

xxxx

### Import Required Libraries

Run the Required Libraries before executing any simulation or forecasting experiment.

In [1]:
import os
os.chdir(r"C:/Users/huma1003/OneDrive - NIQ/DOCUMENTOS IMPORTANTES/Mis_Cosas/MarlijarTM/msvr-master")
from model.Base import Base
from model.MSVR import MSVR
from model.utility import (
    create_dataset,
    create_dataset_antes,
    rmse,
    CustomMSVR,
    create_dataset_rez,
    rezago_sig
)

from scikeras.wrappers import KerasRegressor

from scipy.linalg import orth
from scipy.stats import multivariate_normal

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit,
    train_test_split
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank

import csv
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
file_path = "C:/Users/huma1003/OneDrive - NIQ/DOCUMENTOS IMPORTANTES/Mis_Cosas/MarlijarTM/msvr-master/Resultados_estudio de caso/Profe Sergio/Dats sets.xlsx"
data = pd.read_excel(file_path)
data=data

t=len(data) # Longitud de la serie
k =2 # dimensión del vector Y
p = 1 # Número de retardos
h=1
col=2

# Almacenar resultados
hiperparametros_svr = []
vectores_soporte=[]
train_RMSE_svr = []
test_RMSE_svr = []
train_RMSE_var = []
test_RMSE_var = []
tiempo_var=[]
tiempo_msvr=[]
# Generación de la serie y ajuste de modelos   
series = data.iloc[:, 1:3]
   
#Partición en train y test
train_size = int(len(series) * 0.7)
train, test = series.iloc[:train_size], series.iloc[train_size:]
#PACF
pacf_var1 = pacf(train['RTB3MSY1'], nlags=16)
pacf_var2 = pacf(train['RTN3YSY2'], nlags=16)
banda= 1.96 / np.sqrt(t)  
   

  

fig, axes = plt.subplots(2, 1, figsize=(10, 10))
# Plotear la PACF de Y1
axes[0].stem(range(len(pacf_var1)), pacf_var1, basefmt=" ", use_line_collection=True)
axes[0].axhline(y=banda, color='r', linestyle='--', label=f'Banda superior ({banda:.2f})')
axes[0].axhline(y=-banda, color='r', linestyle='--', label=f'Banda inferior ({-banda:.2f})')
axes[0].set_xlabel('Rezago')
axes[0].set_ylabel('PACF Y1')
axes[0].set_title('PACF de Y1 con Bandas de Confianza')
axes[0].legend()

# Plotear la PACF de Y2
axes[1].stem(range(len(pacf_var2)), pacf_var2, basefmt=" ", use_line_collection=True)
axes[1].axhline(y=banda, color='r', linestyle='--', label=f'Banda superior ({banda:.2f})')
axes[1].axhline(y=-banda, color='r', linestyle='--', label=f'Banda inferior ({-banda:.2f})')
axes[1].set_xlabel('Rezago')
axes[1].set_ylabel('PACF Y2')
axes[1].set_title('PACF de Y2 con Bandas de Confianza')
axes[1].legend()

# Plotear la PACF de Y3
#axes[2].stem(range(len(pacf_var3)), pacf_var3, basefmt=" ", use_line_collection=True)
#axes[2].axhline(y=banda, color='r', linestyle='--', label=f'Banda superior ({banda:.2f})')
#axes[2].axhline(y=-banda, color='r', linestyle='--', label=f'Banda inferior ({-banda:.2f})')
#axes[2].set_xlabel('Rezago')
#axes[2].set_ylabel('PACF Y3')
#axes[2].set_title('PACF de Y3 con Bandas de Confianza')
#axes[2].legend()
# Mostrar los gráficos
#plt.tight_layout()
#plt.show()

  
rez=2   
# Ajustar modelo VAR
start_time = time.time()
model_var = VAR(train)
results_var = model_var.fit(maxlags=5, ic='aic')
lag_order = results_var.k_ar

modelo_var_train = []
modelo_var_test = []
    
#Predicciones para train
train_pred = results_var.fittedvalues
# Predicciones para test
test_pred=[]
input_data = train.values[-rez:]

for i in range(len(test)):
    pred = results_var.forecast(y=input_data, steps=h)
    test_pred.append(pred[0])
    input_data = np.vstack([input_data[1:], test.values[i:i+1]])

test_pred = pd.DataFrame(test_pred) 
#Sacar rmse     
#train_rmse_var = np.sqrt(np.mean((train.iloc[lag_order:, 0] - train_pred.iloc[:, 0])**2 + (train.iloc[lag_order:, 1] - train_pred.iloc[:, 1])**2))
train_rmse_var = np.sqrt(np.mean((train.iloc[lag_order:, 0] - train_pred.iloc[:, 0])**2 + (train.iloc[lag_order:, 1] - train_pred.iloc[:, 1])**2))


# Cálculo del RMSE para el conjunto de prueba
#test_rmse_var = np.sqrt(np.mean((test.iloc[:, 0] - test_pred.iloc[:, 0])**2 + (test.iloc[:, 1] - test_pred.iloc[:, 1])**2))
test_rmse_var = np.sqrt(np.mean((test.iloc[:, 0] - test_pred.iloc[:, 0])**2 + (test.iloc[:, 1] - test_pred.iloc[:, 1])**2))

print(test_RMSE_var)
import pandas as pd

train_pred_df = pd.DataFrame(train_pred, columns=['Predicción_Var1', 'Predicción_Var2' ])
test_pred_df = pd.DataFrame(test_pred, columns=['Predicción_Var1', 'Predicción_Var2'])

# Concatenar los DataFrames
train_combined = pd.concat([train.reset_index(drop=True), train_pred.reset_index(drop=True)], axis=1)
test_combined = pd.concat([test.reset_index(drop=True), test_pred.reset_index(drop=True)], axis=1)

# Guardar en archivos CSV
train_combined.to_csv('train_with_predictions_sergio_Var.csv', index=False)
test_combined.to_csv('test_with_predictions_sergio_var.csv', index=False)






print("Termine de ajustar modelo VAR")
end_time = time.time()
execution_time = end_time - start_time
tiempo_var.append(execution_time)

# Ajustar modelo SVR
start_time = time.time()
fechas = pd.DataFrame(list(range(len(series))))
total = pd.concat([fechas,series], axis=1).values
dim=len(total)

#Construcción de la base de datos
data=Base(total)
data= data.base
#Creamos la base de datos
dataset = create_dataset_rez(data,dim,h,col,rez)
X, Y = dataset[:, :(0 - h*2)], dataset[:, (0-h*2):]
#Train y test
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)

#normalizados
scaler_X = StandardScaler()
scaler_y = StandardScaler()

scaler_X.fit(X_train)
scaler_y.fit(y_train)

   
X_train_nor = scaler_X.transform(X_train)
X_test_nor = scaler_X.transform(X_test)
y_train_nor = scaler_y.transform(y_train)
y_test_nor = scaler_y.transform(y_test)

pipe = Pipeline([
    ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
])

hyperparameters = {
    #'MSVR__kernel': ['poly'],
    'MSVR__kernel': ['poly','rbf','linear'],
    'MSVR__degree': [2,5],
    #'MSVR__degree': [1],
    'MSVR__gamma': [0.5,1],
    'MSVR__coef0': [0.1,0.5,1],
    'MSVR__C': [5,9,11,13],
    'MSVR__epsilon':[1,2], 
}

bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise')
   
best_model = bm.fit(X_train_nor, y_train_nor)
best_params = bm.best_params_

msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
            epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
            degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)

msvr.fit(X_train_nor, y_train_nor)
  
 
trainPred_svr_nor = msvr.predict(X_train_nor)
testPred_svr_nor = msvr.predict(X_test_nor)

trainPred_svr  = scaler_y.inverse_transform(trainPred_svr_nor)
testPred_svr  = scaler_y.inverse_transform(testPred_svr_nor)
train_rmse_svr = rmse(y_train, trainPred_svr)
test_rmse_svr = rmse(y_test, testPred_svr)
train_pred_df = pd.DataFrame(trainPred_svr, columns=['Predicción_Var1', 'Predicción_Var2'])
test_pred_df = pd.DataFrame(testPred_svr, columns=['Predicción_Var1', 'Predicción_Var2'])

# Concatenar los DataFrames
train_combined = pd.concat([train.reset_index(drop=True), train_pred_df.reset_index(drop=True)], axis=1)
test_combined = pd.concat([test.reset_index(drop=True), test_pred_df.reset_index(drop=True)], axis=1)

# Guardar en archivos CSV
train_combined.to_csv('train_with_prediction_svr_1_sergio.csv', index=False)
test_combined.to_csv('test_with_predictions_svr_1_sergio.csv', index=False)

hiperparametros_svr.append(best_params)
vectores_soporte.append(msvr.NSV)
print("SVR Best params:", msvr.NSV/t)
train_RMSE_svr.append(train_rmse_svr)
test_RMSE_svr.append(test_rmse_svr)

end_time = time.time()
execution_time = end_time - start_time
tiempo_msvr.append(execution_time)


print("SVR Best params:", best_params)
print("VAR Train RMSE:", train_rmse_var, "Test RMSE:", test_rmse_var)
print("SVR Train RMSE:", train_rmse_svr, "Test RMSE:", test_rmse_svr)

#Pronóstico de persistencia


# Crear conjunto de entrenamiento y prueba
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

# Pronóstico de persistencia
def persistence_forecast(train, test):
    predictions = pd.concat([train.iloc[[-1]], test.iloc[:-1]])  # Persistencia para todas las columnas
    return predictions

# Generar predicciones
predictions = persistence_forecast(train, test)

# Evaluar el modelo
test_rmse_per = rmse(test.iloc[:, 1:3], predictions.iloc[:, 1:3])
print(test_rmse_per)

# Visualizar resultados
plt.figure(figsize=(12, 6))

# Serie 1 y Serie 2 en un solo gráfico
plt.plot(test.index, test['u_comp'], label='Serie 1 (Real)', color='blue')
plt.plot(test.index, predictions['u_comp'], label='u_comp', linestyle='dashed', color='orange')
plt.plot(test.index, test['v_comp'], label='Serie 2 (Real)', color='green')
plt.plot(test.index, predictions['v_comp'], label='Pronóstico Serie 2', linestyle='dashed', color='red')

plt.title('Series Originales y Pronóstico de Persistencia')
plt.legend()
plt.show()

import matplotlib.pyplot as plt

# Crear una figura con dos subgráficos (uno por serie)
plt.figure(figsize=(12, 6))

# Gráfico para Serie 1
plt.subplot(2, 1, 1)  # 2 filas, 1 columna, primer gráfico
plt.plot(test.index, test['u_comp'], label='Serie 1 (Real)', color='blue')
plt.plot(test.index, predictions['u_comp'], label='Pronóstico u_comp', linestyle='dashed', color='orange')
plt.title('Serie 1 - Pronóstico de Persistencia')
plt.legend()

# Gráfico para Serie 2
plt.subplot(2, 1, 2)  # 2 filas, 1 columna, segundo gráfico
plt.plot(test.index, test['v_comp'], label='v_comp (Real)', color='green')
plt.plot(test.index, predictions['v_comp'], label='Pronóstico v_comp', linestyle='dashed', color='red')
plt.title('Serie 2 - Pronóstico de Persistencia')
plt.legend()

# Ajuste del diseño para que no se superpongan los elementos
plt.tight_layout()

# Mostrar la gráfica
plt.show()
